In [0]:
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

# Read bronze crypto data
print("📖 Reading bronze crypto data...")
bronze_df = spark.table("financial_market.bronze.crypto")

print(f"Bronze records: {bronze_df.count()}")

# Display initial data quality overview
print("\n📊 Initial Data Quality Check:")
print(f"Total records: {bronze_df.count()}")
print(f"Null prices: {bronze_df.filter(F.col('price_usd').isNull()).count()}")
print(f"Null symbols: {bronze_df.filter(F.col('symbol').isNull()).count()}")
print(f"Null market_cap: {bronze_df.filter(F.col('market_cap').isNull()).count()}")

display(bronze_df.limit(5))

In [0]:
# ============================
# DATA CLEANING PIPELINE
# ============================

print("✨ Starting data cleaning pipeline...\n")

# Step 1: Remove null or invalid symbols
print("🧹 Step 1: Removing null/empty symbols...")
cleaned_df = bronze_df.filter(
    (F.col("symbol").isNotNull()) & 
    (F.trim(F.col("symbol")) != "")
)
print(f"  Records after removing null symbols: {cleaned_df.count()}")

# Step 2: Standardize symbol and name
print("📝 Step 2: Standardizing symbol and name...")
cleaned_df = cleaned_df \
    .withColumn("symbol", F.upper(F.trim(F.col("symbol")))) \
    .withColumn("name", F.trim(F.col("name")))

# Step 3: Filter out invalid prices (null, zero, or negative)
print("💰 Step 3: Filtering invalid prices...")
cleaned_df = cleaned_df.filter(
    (F.col("price_usd").isNotNull()) & 
    (F.col("price_usd") > 0)
)
print(f"  Records after price validation: {cleaned_df.count()}")

# Step 4: Handle null values in numeric columns (fill with 0)
print("🔢 Step 4: Handling null values in numeric columns...")
cleaned_df = cleaned_df \
    .fillna(0, subset=["market_cap", "volume_24h", "price_change_24h_pct", "rank"])

# Step 5: Add derived columns
print("➕ Step 5: Adding derived columns...")
cleaned_df = cleaned_df \
    .withColumn("fetch_date", F.to_date(F.col("fetch_timestamp"))) \
    .withColumn("is_stablecoin", 
                F.when(F.col("symbol").isin("USDT", "USDC", "BUSD", "DAI", "TUSD"), True)
                .otherwise(False)) \
    .withColumn("market_cap_billions", F.round(F.col("market_cap") / 1e9, 2)) \
    .withColumn("volume_24h_millions", F.round(F.col("volume_24h") / 1e6, 2))

# Step 6: Remove duplicates - keep latest record per symbol
print("🗑️ Step 6: Removing duplicates (keeping latest per symbol)...")
window_spec = Window.partitionBy("symbol").orderBy(F.col("fetch_timestamp").desc())
cleaned_df = cleaned_df \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

print(f"  Records after deduplication: {cleaned_df.count()}")

# Step 7: Add data quality flags
print("✅ Step 7: Adding data quality flags...")
cleaned_df = cleaned_df \
    .withColumn("has_market_cap", F.when(F.col("market_cap") > 0, True).otherwise(False)) \
    .withColumn("has_volume", F.when(F.col("volume_24h") > 0, True).otherwise(False)) \
    .withColumn("data_quality_score", 
                F.when(F.col("has_market_cap") & F.col("has_volume"), 100)
                .when(F.col("has_market_cap") | F.col("has_volume"), 50)
                .otherwise(0))

# Step 8: Add processing metadata
print("📋 Step 8: Adding processing metadata...")
cleaned_df = cleaned_df \
    .withColumn("processed_timestamp", F.current_timestamp()) \
    .withColumn("processing_date", F.current_date())

print(f"\n✅ Cleaning completed! Final record count: {cleaned_df.count()}\n")

# Display cleaned data sample
display(cleaned_df.orderBy(F.col("market_cap").desc()).limit(10))

In [0]:
# ============================
# SAVE TO SILVER LAYER
# ============================

print("📦 Saving cleaned data to silver layer...\n")

# Create silver schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS financial_market.silver")

# Define silver table name
silver_table = "financial_market.silver.crypto"

# Write to silver table with overwrite mode (for full refresh)
# Use append mode if you want incremental loads
cleaned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print(f"✅ Data successfully saved to {silver_table}")

# Verify the save
silver_count = spark.table(silver_table).count()
print(f"\n📊 Silver table record count: {silver_count}")

# Show summary statistics
print("\n📊 Summary Statistics:")
summary_df = spark.table(silver_table).select(
    F.count("*").alias("total_records"),
    F.countDistinct("symbol").alias("unique_cryptos"),
    F.round(F.avg("price_usd"), 2).alias("avg_price_usd"),
    F.round(F.sum("market_cap_billions"), 2).alias("total_market_cap_billions"),
    F.sum(F.when(F.col("is_stablecoin") == True, 1).otherwise(0)).alias("stablecoins_count"),
    F.round(F.avg("data_quality_score"), 2).alias("avg_quality_score")
)

display(summary_df)